### Σύγκριση Temperature


### Μέρος Α
#### Σύγκριση ασαφούς prompt
Ασαφής prompt: "a sort function"
##### **Αποτέλεσμα με temperature 0.2:** 
```python 
def sort_list(items: list, reverse: bool = False) -> list:
    """Sort a list of comparable elements in ascending or descending order.

    Args:
        items (list): List of elements to sort. Elements must be mutually comparable.
        reverse (bool): If True, sort in descending order. Defaults to False.

    Returns:
        list: A new sorted list.

    Raises:
        TypeError: If items is not a list or contains non-comparable elements.

    Example:
        >>> sort_list([3, 1, 2])
        [1, 2, 3]
        >>> sort_list([3, 1, 2], reverse=True)
        [3, 2, 1]
        >>> sort_list([])
        []
    """
    if not isinstance(items, list):
        raise TypeError(f"Expected a list, got {type(items).__name__}.")

    if not isinstance(reverse, bool):
        raise TypeError(f"Expected bool for reverse, got {type(reverse).__name__}.")

    if len(items) == 0:
        return []

    try:
        return sorted(items, reverse=reverse)
    except TypeError as exc:
        raise TypeError(
            "List contains elements that cannot be compared to each other."
        ) from exc
```
##### **Αποτέλεσμα με temperature 0.9:**
```python
def sort_list(items: list, reverse: bool = False) -> list:
    """Sort a list of comparable elements in ascending or descending order.

    Args:
        items (list): List of elements to sort. Elements must be mutually
            comparable (e.g., all ints, all strings).
        reverse (bool): If True, sort in descending order. Defaults to False.

    Returns:
        list: A new sorted list.

    Raises:
        TypeError: If *items* is not a list or contains non-comparable types.

    Example:
        >>> sort_list([3, 1, 2])
        [1, 2, 3]
        >>> sort_list([3, 1, 2], reverse=True)
        [3, 2, 1]
        >>> sort_list([])
        []
    """
    if not isinstance(items, list):
        raise TypeError(
            f"Expected a list, got {type(items).__name__}."
        )

    if not items:
        return []

    # Verify that all elements are mutually comparable by checking
    # they share a common comparable base or by attempting a sort.
    try:
        sorted_items = sorted(items, reverse=reverse)
    except TypeError as exc:
        raise TypeError(
            "List contains elements that cannot be compared to each other."
        ) from exc

    return sorted_items
```

### Συμπέρασμα σύγκρισης:
Με temperature 0.9 διακρίνω ότι έχει ανακρίβειες στον σχολιασμό πάνω απο το try-except. <<Γράφει κάποια πράγματα που δεν τα κάνει τελικά>>. Κατα τα άλλα εξάγει μια συνάρτηση καθ'όλα λειτουργική. Το πιό ενδιαφέρων είναι και πάλι ότι λόγο του ότι κάναμε καλό prompt engineering στο προγραμματάκι μας ακόμα και μιά τόσο υψηλή θερμοκρασία παράγει σχετικά προβλεπόμενα αποτελέσματα. 
Έκανα ένα τεστ να τρέξω την ίδια προτροπή με "θερμοκρασία" 0.9 εκτός του προγραμματός μας και πείρα αυτό το αποτέλεσμα κώδικα (αφαιρώντας την πολυλογία γύρο απο τον κώδικα):
``` Python
   import threading
import time

def sleep_sort(data):
    sorted_list = []
    threads = []

    def add_to_list(n): 
        # We wait for 'n' seconds before adding to the list.
        time.sleep(n)
        sorted_list.append(n)

    # Launch a thread for every single element
    for x in data:
        t = threading.Thread(target=add_to_list, args=(x,))
        threads.append(t)
        t.start()

    # Wait for all "sleepers" to wake up
    for t in threads:
        t.join()

    return sorted_list

# Example usage:
unsorted = [0.5, 0.2, 0.8, 0.1, 0.3]
print(f"Original: {unsorted}")
print(f"Sorted via 'napping': {sleep_sort(unsorted)}")
```
##### Τα αποτελέσματα είναι σαφώς ευδιάκριτα, εδώ μας έδωσε κάτι τελείως διαφορετικό. (προσωπικά το έβαλα σε LLM για να με εξειγήσει τι γίνεται σε αυτήν την συνάρτηση 😅.) Και βλέπουμε επίσης ότι δεν έχει τύπους, δεν κάνει error handling, δεν έχει docstrings κ.λ.π

### Μέρος Β
#### Σύγκριση συγκεκριμένου prompt
Το prompt: "a function that implements quicksort with in-place partitioning for a list of integers"
##### **Αποτέλεσμα με temperature 0.2:** 
```python 
def quicksort(arr: list[int]) -> list[int]:
    """Sort a list of integers in-place using the quicksort algorithm.

    Uses Lomuto partition scheme for in-place partitioning. The original
    list is modified and also returned for convenience.

    Args:
        arr (list[int]): The list of integers to sort.

    Returns:
        list[int]: The same list, sorted in ascending order.

    Raises:
        TypeError: If input is not a list or contains non-integer elements.

    Example:
        >>> quicksort([3, 1, 4, 1, 5, 9, 2, 6])
        [1, 1, 2, 3, 4, 5, 6, 9]
        >>> quicksort([])
        []
        >>> quicksort([42])
        [42]
    """
    if not isinstance(arr, list):
        raise TypeError(f"Expected a list, got {type(arr).__name__}.")

    for idx, item in enumerate(arr):
        if not isinstance(item, int) or isinstance(item, bool):
            raise TypeError(
                f"All elements must be integers. "
                f"Found {type(item).__name__} at index {idx}."
            )

    def _partition(lo: int, hi: int) -> int:
        """Lomuto partition around the last element as pivot.

        Args:
            lo (int): Lower bound index (inclusive).
            hi (int): Upper bound index (inclusive).

        Returns:
            int: Final index of the pivot element.
        """
        pivot = arr[hi]
        i = lo - 1
        for j in range(lo, hi):
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i + 1], arr[hi] = arr[hi], arr[i + 1]
        return i + 1

    def _quicksort(lo: int, hi: int) -> None:
        """Recursively sort the sub-array arr[lo..hi] in place.

        Args:
            lo (int): Lower bound index (inclusive).
            hi (int): Upper bound index (inclusive).
        """
        if lo < hi:
            pivot_idx = _partition(lo, hi)
            _quicksort(lo, pivot_idx - 1)
            _quicksort(pivot_idx + 1, hi)

    if len(arr) > 1:
        _quicksort(0, len(arr) - 1)

    return arr
```
##### **Αποτέλεσμα με temperature 0.9:** 
```Python
def quicksort(arr: list[int]) -> list[int]:
    """Sort a list of integers in-place using the quicksort algorithm.

    Uses Lomuto partition scheme for in-place partitioning. The original
    list is modified and also returned for convenience.

    Args:
        arr (list[int]): The list of integers to sort.

    Returns:
        list[int]: The same list, sorted in ascending order.

    Raises:
        TypeError: If input is not a list or contains non-integer elements.

    Example:
        >>> quicksort([3, 1, 4, 1, 5, 9, 2, 6])
        [1, 1, 2, 3, 4, 5, 6, 9]
        >>> quicksort([])
        []
        >>> quicksort([42])
        [42]
    """
    if not isinstance(arr, list):
        raise TypeError(f"Expected a list, got {type(arr).__name__}.")

    for idx, item in enumerate(arr):
        if not isinstance(item, int) or isinstance(item, bool):
            raise TypeError(
                f"All elements must be integers. "
                f"Found {type(item).__name__} at index {idx}."
            )

    def _partition(lo: int, hi: int) -> int:
        """Partition arr[lo..hi] in-place using arr[hi] as pivot.

        Args:
            lo (int): Lower bound index (inclusive).
            hi (int): Upper bound index (inclusive), also the pivot index.

        Returns:
            int: Final index of the pivot element.
        """
        pivot = arr[hi]
        i = lo - 1
        for j in range(lo, hi):
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i + 1], arr[hi] = arr[hi], arr[i + 1]
        return i + 1

    def _quicksort(lo: int, hi: int) -> None:
        """Recursively sort arr[lo..hi] in-place.

        Args:
            lo (int): Lower bound index (inclusive).
            hi (int): Upper bound index (inclusive).
        """
        if lo < hi:
            pivot_idx = _partition(lo, hi)
            _quicksort(lo, pivot_idx - 1)
            _quicksort(pivot_idx + 1, hi)

    if len(arr) > 1:
        _quicksort(0, len(arr) - 1)

    return arr
```

### Συμπέρασμα σύγκρισης:
Με temperature 0.9 ουσιαστικά άλλαξαν ελαφρός κάποιες λέξεις στο docstring. Φαίνετε ότι με ένα ακριβές prompt και με ένα καλά στημένο prompt enginnering εσωτερικά στο προγραμματάκι μας η υψηλότερη "θερμοκρασία" δεν παράγει πολύ διαφορετικά αποτελέσματα.  